# 单向与双向 LSTM 对照

固定词表、序列长度、训练数据和其他训练参数，比较单向与双向 LSTM。每个模型使用相同的两个随机种子；按验证损失选择各次运行的最佳轮次。测试集不参与实验。
双向模型每个方向的隐藏维度仍为 128，因此参数量和计算量高于单向模型。


In [ ]:
# 加载训练集、验证集和实验依赖
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.lstm import LSTMClassifier, BidirectionalLSTMClassifier
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import build_vocabulary, SentimentDataset, TOKEN_PATTERN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_data = pd.read_csv(CLEANED_DIR / "train.csv")
validation_data = pd.read_csv(CLEANED_DIR / "val.csv")

max_words = 20_000
max_length = 512
batch_size = 32
max_epochs = 20
patience = 4
learning_rate = 0.001
gradient_clip = 1.0

vocabulary = build_vocabulary(train_data["text"], max_words=max_words)
train_dataset = SentimentDataset(train_data, vocabulary, max_length)
validation_dataset = SentimentDataset(validation_data, vocabulary, max_length)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Train: {len(train_dataset):,} | Validation: {len(validation_dataset):,}")
print(f"Vocabulary: {len(vocabulary):,}")


In [ ]:
# 定义训练与验证流程
def train_one_epoch(model, data_loader, optimizer, loss_function):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for token_ids, lengths, labels in data_loader:
        token_ids = token_ids.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(token_ids, lengths)
        loss = loss_function(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate_model(model, data_loader, loss_function):
    model.eval()
    total_loss = 0.0
    total = 0
    y_true = []
    y_prediction = []

    with torch.no_grad():
        for token_ids, lengths, labels in data_loader:
            token_ids = token_ids.to(device)
            labels = labels.to(device)
            logits = model(token_ids, lengths)
            loss = loss_function(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            y_true.extend(labels.cpu().tolist())
            y_prediction.extend(logits.argmax(dim=1).cpu().tolist())

    return total_loss / total, classification_metrics(y_true, y_prediction)


In [ ]:
# 运行一组结构和种子，并按最低验证损失保存检查点
def run_experiment(architecture, random_seed):
    torch.manual_seed(random_seed)
    generator = torch.Generator().manual_seed(random_seed)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, generator=generator,
    )

    model_config = {
        "vocabulary_size": len(vocabulary),
        "embedding_dim": 128,
        "hidden_size": 128,
        "padding_index": vocabulary["<PAD>"],
        "dropout": 0.5,
    }
    if architecture == "Unidirectional":
        model = LSTMClassifier(**model_config).to(device)
    elif architecture == "Bidirectional":
        model = BidirectionalLSTMClassifier(**model_config).to(device)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history = []
    checkpoint_path = CHECKPOINTS_DIR / f"lstm_{architecture.lower()}_seed{random_seed}_best.pt"

    for epoch in range(1, max_epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, loss_function)
        validation_loss, metrics = evaluate_model(model, validation_loader, loss_function)

        history.append({
            "ARCHITECTURE": architecture, "SEED": random_seed, "EPOCH": epoch,
            "TRAIN LOSS": train_loss, "TRAIN ACCURACY": train_accuracy,
            "VALIDATION LOSS": validation_loss, "VALIDATION ACCURACY": metrics["ACCURACY"],
            "VALIDATION F1": metrics["F1"],
        })
        print(
            f"{architecture} | Seed {random_seed} | Epoch {epoch} | "
            f"Train loss {train_loss:.4f} | Validation loss {validation_loss:.4f} | "
            f"Validation F1 {metrics['F1']:.4f}",
            flush=True,
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            epochs_without_improvement = 0
            best_result = {
                "ARCHITECTURE": architecture, "SEED": random_seed, "BEST EPOCH": epoch,
                "PARAMETERS": parameter_count, "VALIDATION LOSS": validation_loss, **metrics,
            }
            torch.save({
                "model_state_dict": model.state_dict(),
                "model_config": model_config,
                "architecture": architecture,
                "vocabulary": vocabulary,
                "max_length": max_length,
                "token_pattern": TOKEN_PATTERN.pattern,
                "lowercase": True,
                "random_seed": random_seed,
                "epoch": epoch,
                "validation_loss": validation_loss,
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping.", flush=True)
                break

        # 每轮保存进度，避免中断后丢失训练曲线
        pd.DataFrame(history).to_csv(
            TABLES_DIR / f"lstm_{architecture.lower()}_seed{random_seed}_history.csv", index=False,
        )

    pd.DataFrame(history).to_csv(
        TABLES_DIR / f"lstm_{architecture.lower()}_seed{random_seed}_history.csv", index=False,
    )
    return best_result


## 单一种子结构对照

复用现有单向 LSTM、种子 42 的检查点，再在相同种子下训练双向 LSTM。以验证损失最低的轮次比较两种结构，同时记录 F1。


In [ ]:
# 加载现有单向基线并核对数据处理配置
baseline_checkpoint = torch.load(
    CHECKPOINTS_DIR / "lstm_best.pt", map_location=device, weights_only=True,
)
assert baseline_checkpoint["vocabulary"] == vocabulary
assert baseline_checkpoint["token_pattern"] == TOKEN_PATTERN.pattern
assert baseline_checkpoint["max_length"] == max_length
assert baseline_checkpoint["random_seed"] == 42

baseline_model = LSTMClassifier(**baseline_checkpoint["model_config"]).to(device)
baseline_model.load_state_dict(baseline_checkpoint["model_state_dict"])
baseline_loss, baseline_metrics = evaluate_model(baseline_model, validation_loader, nn.CrossEntropyLoss())

baseline_result = {
    "ARCHITECTURE": "Unidirectional",
    "SEED": 42,
    "BEST EPOCH": baseline_checkpoint["epoch"],
    "PARAMETERS": sum(parameter.numel() for parameter in baseline_model.parameters()),
    "VALIDATION LOSS": baseline_loss,
    **baseline_metrics,
}
experiment_results = [baseline_result]

print("Existing baseline:", baseline_result)


In [ ]:
# 训练双向 LSTM 并比较
bidirectional_result = run_experiment("Bidirectional", random_seed=42)
experiment_results.append(bidirectional_result)
architecture_comparison = pd.DataFrame(experiment_results)

display(architecture_comparison.round(4))
architecture_comparison.to_csv(TABLES_DIR / "lstm_architecture_results.csv", index=False)


## 相同随机种子复验

再用种子 123 分别训练单向和双向模型。两次运行均采用相同的早停规则，比较两种结构在相同随机种子下的表现。两次种子只能提供初步的稳定性观察。


In [ ]:
# 对两种结构使用同一新种子
for architecture in ["Unidirectional", "Bidirectional"]:
    result = run_experiment(architecture, random_seed=123)
    experiment_results.append(result)
    pd.DataFrame(experiment_results).to_csv(TABLES_DIR / "lstm_architecture_results.csv", index=False)

all_results = pd.DataFrame(experiment_results)
display(all_results.sort_values(["ARCHITECTURE", "SEED"]).round(4))


In [ ]:
# 汇总每种结构的验证结果
stability_summary = (
    all_results.groupby("ARCHITECTURE")
    .agg(
        RUNS=("F1", "count"),
        LOSS_MEAN=("VALIDATION LOSS", "mean"),
        F1_MEAN=("F1", "mean"),
        F1_STD=("F1", "std"),
        F1_MIN=("F1", "min"),
        F1_MAX=("F1", "max"),
        ACCURACY_MEAN=("ACCURACY", "mean"),
        PARAMETERS=("PARAMETERS", "first"),
    )
    .reset_index()
)

display(stability_summary.round(4))
stability_summary.to_csv(TABLES_DIR / "lstm_architecture_stability.csv", index=False)


In [ ]:
# 展示相同随机种子下的结构对照
import matplotlib.pyplot as plt
from sentiment_analysis.paths import FIGURES_DIR

figure, axis = plt.subplots(figsize=(7, 5))

for random_seed, data in all_results.groupby("SEED"):
    data = data.set_index("ARCHITECTURE").loc[["Unidirectional", "Bidirectional"]]
    axis.plot(data.index, data["F1"], marker="o", label=f"Seed {random_seed}")
    for architecture, score in data["F1"].items():
        label_offset = 8 if random_seed == 42 else -14
        axis.annotate(f"{score:.3f}", (architecture, score), xytext=(0, label_offset),
                      textcoords="offset points", ha="center")

axis.set_xlabel("LSTM architecture")
axis.set_ylabel("Validation F1")
axis.set_title("LSTM architecture comparison")
axis.set_ylim(0, 1)
axis.legend()
figure.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURES_DIR / "lstm_architecture_comparison.png", dpi=150)
plt.show()

## 阶段性观察

按最低验证损失选取每次运行的检查点后，两个种子下的单向 LSTM 平均验证 F1 为 0.8492，双向 LSTM 为 0.8519；平均验证损失分别为 0.3632 和 0.3604。差距较小，不能据此断定双向结构普遍更优。

双向模型每个方向的隐藏维度与单向模型相同，因此参数量由 2,692,610 增至 2,824,962；本次并未控制参数量。可以暂将双向结构作为后续分析的候选，同时保留原单向基线。

以上是两个种子、同一验证集上的探索性比较；尚未使用测试集。后续融合实验如果使用双向模型，应重新生成预测并明确记录模型选择。